# Empirical Analysis of Technical Indicators and Macro Factors on Nifty Equities
### A Panel Data Econometrics Study (2014 - 2024)

This notebook implements a quantitative research pipeline to study the empirical relationship between common technical trading factors and stock returns for Nifty 200 constituents over a 10-year historical window.

#### Methodology Overview:
1. **Ingestion & Filtering**: Download constituent lists and historical daily pricing from Yahoo Finance. Keep assets with sufficient trading histories (>= 85% data density) to avoid selection/survivorship bias.
2. **Feature Engineering**: Calculate Daily Log Returns, rolling Historical Volatility, rolling Momentum, Price-to-Moving-Average ratio, and a custom Relative Strength Index (RSI-14).
3. **Macro Controls**: Merge India VIX (systemic volatility proxy) and Nifty 50 Index returns (market proxy) as control variables.
4. **Outlier Mitigation**: Winsorize continuous variables at the 1st and 99th percentiles to avoid extreme value bias.
5. **Panel Regression Models**: Estimate Pooled OLS, Fixed Effects (Entity), Random Effects, and First Difference architectures.

In [ ]:
import os
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from nsepython import nse_eq_symbols
from linearmodels.panel import PanelOLS, RandomEffects
import statsmodels.api as sm

print("Libraries successfully imported.")

## 1. Data Acquisition & Ingestion
We fetch the list of Nifty 200 constituents from the National Stock Exchange of India (NSE) index repository, map them to Yahoo Finance tickers, and download daily pricing data from 2014 through 2024.

In [ ]:
print("Fetching Nifty 200 constituents list...")
url = "https://archives.nseindia.com/content/indices/ind_nifty200list.csv"
nifty200_df = pd.read_csv(url)
companies = (nifty200_df["Symbol"].astype(str) + ".NS").tolist()

print(f"Total constituents to download: {len(companies)}")

# Ingest pricing data
all_data = []
failed = []

for ticker in companies:
    try:
        df = yf.download(
            ticker,
            start="2014-01-01",
            end="2025-01-01",
            interval="1d",
            auto_adjust=True,
            progress=False
        )
        if df.empty:
            failed.append(ticker)
            continue
            
        df = df.reset_index()
        df["Firm"] = ticker
        
        # Flatten MultiIndex columns
        df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
        df = df[["Date", "Firm", "Close", "Open", "High", "Low", "Volume"]]
        all_data.append(df)
    except Exception as e:
        failed.append(ticker)

panel_df = pd.concat(all_data, ignore_index=True)
print(f"Successfully downloaded {panel_df['Firm'].nunique()} firms.")

## 2. Data Cleaning & Balanced Panel Filters
To establish a reliable econometric panel, we:
1. Sort the observations strictly by `Firm` and `Date`.
2. Exclude firms with extremely short histories (fewer than 1,500 trading days) to eliminate bias from newly listed stocks.

In [ ]:
# Pre-selection filter
obs_count = panel_df.groupby("Firm").size()
valid_firms_density = obs_count[obs_count >= 1500].index
panel_df = panel_df[panel_df["Firm"].isin(valid_firms_density)]

# Sort panel
panel_df = panel_df.sort_values(by=["Firm", "Date"]).reset_index(drop=True)
print(f"Remaining firms after initial density filter: {panel_df['Firm'].nunique()}")

## 3. Technical Feature Engineering
We calculate standard financial factors from raw closing prices:
* **Log Return**: Daily logarithmic price difference.
* **Volatility**: 21-day rolling standard deviation of daily log returns.
* **Momentum**: 21-day rolling returns capturing immediate asset trend.
* **MA Ratio**: Price divided by its 21-day rolling average.
* **RSI (Relative Strength Index)**: Built from scratch to track rolling gains vs. losses over a 14-day window.

In [ ]:
# Log returns
panel_df["Log_Return"] = panel_df.groupby("Firm")["Close"].transform(
    lambda x: np.log(x / x.shift(1))
)

# 21-day rolling volatility
panel_df["Volatility"] = panel_df.groupby("Firm")["Log_Return"].transform(
    lambda x: x.rolling(21).std()
)

# 21-day momentum
panel_df["Momentum"] = panel_df.groupby("Firm")["Close"].transform(
    lambda x: (x / x.shift(21)) - 1
)

# MA Ratio
ma21 = panel_df.groupby("Firm")["Close"].transform(
    lambda x: x.rolling(21).mean()
)
panel_df["MA_Ratio"] = panel_df["Close"] / ma21

# RSI-14
delta = panel_df.groupby("Firm")["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.groupby(panel_df["Firm"]).transform(lambda x: x.rolling(14).mean())
avg_loss = loss.groupby(panel_df["Firm"]).transform(lambda x: x.rolling(14).mean())

rs = avg_gain / avg_loss
panel_df["RSI"] = 100 - (100 / (1 + rs))

panel_df[["Log_Return", "Volatility", "Momentum", "MA_Ratio", "RSI"]].describe()

## 4. Outlier Mitigation & Macroeconomic Controls
1. **Winsorization**: Tail outliers can disproportionately influence regression weights. We winsorize our key variables at the 1% and 99% thresholds.
2. **Market Proxy**: Download Nifty 50 returns (`^NSEI`) to control for overall systematic market moves.
3. **VIX Control**: Fetch the India VIX (`^INDIAVIX`) to control for macroeconomic risk sentiment.

In [ ]:
# Winsorize tail variables
for col in ["Log_Return", "Momentum", "Volatility"]:
    lower_bound = panel_df[col].quantile(0.01)
    upper_bound = panel_df[col].quantile(0.99)
    panel_df[col] = np.clip(panel_df[col], lower_bound, upper_bound)

# Download Nifty 50
nifty_idx = yf.download("^NSEI", start="2014-01-01", end="2025-01-01", interval="1d", auto_adjust=True, progress=False)
nifty_idx = nifty_idx.reset_index()
nifty_idx.columns = [col[0] if isinstance(col, tuple) else col for col in nifty_idx.columns]
nifty_idx["Market_Return"] = np.log(nifty_idx["Close"] / nifty_idx["Close"].shift(1))
nifty_idx = nifty_idx[["Date", "Market_Return"]]

panel_df = panel_df.merge(nifty_idx, on="Date", how="left")

# Download India VIX
vix_idx = yf.download("^INDIAVIX", start="2014-01-01", end="2025-01-01", interval="1d", auto_adjust=True, progress=False)
vix_idx = vix_idx.reset_index()
vix_idx.columns = [col[0] if isinstance(col, tuple) else col for col in vix_idx.columns]
vix_idx = vix_idx[["Date", "Close"]].rename(columns={"Close": "VIX"})

panel_df = panel_df.merge(vix_idx, on="Date", how="left")

# Drop NaN values from rolling/shifts
panel_df = panel_df.dropna()

# Enforce high-coverage Balanced Panel threshold (>= 85% coverage of dates)
total_dates = panel_df["Date"].nunique()
obs_per_firm = panel_df.groupby("Firm").size()
coverage_ratio = obs_per_firm / total_dates
valid_firms_coverage = coverage_ratio[coverage_ratio >= 0.85].index
panel_df = panel_df[panel_df["Firm"].isin(valid_firms_coverage)]

print(f"Final panel structure: {panel_df['Firm'].nunique()} firms, {len(panel_df)} observations.")

## 5. Exploratory Data Analysis & Plots
Let's explore correlations and plot sector risk-return behaviors.

In [ ]:
# Setup plots folder
os.makedirs("plots", exist_ok=True)

# Correlation Matrix Heatmap
corr_matrix = panel_df[["Log_Return", "Volatility", "Momentum", "MA_Ratio", "RSI", "Market_Return", "VIX"]].corr()
plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix of Panel Variables")
plt.show()

# Returns Distribution Hist
plt.figure(figsize=(10, 5))
plt.hist(panel_df["Log_Return"], bins=100, color='royalblue', edgecolor='k', alpha=0.7)
plt.title("Histogram of Winsorized Log Daily Returns")
plt.show()

# Sector Profile Map
sector_map = {
    "HDFCBANK.NS": "Banking", "ICICIBANK.NS": "Banking", "SBIN.NS": "Banking", "AXISBANK.NS": "Banking", "KOTAKBANK.NS": "Banking",
    "TCS.NS": "IT", "INFY.NS": "IT", "WIPRO.NS": "IT", "HCLTECH.NS": "IT", "TECHM.NS": "IT",
    "HINDUNILVR.NS": "FMCG", "ITC.NS": "FMCG", "NESTLEIND.NS": "FMCG", "BRITANNIA.NS": "FMCG",
    "SUNPHARMA.NS": "Pharma", "DRREDDY.NS": "Pharma", "CIPLA.NS": "Pharma",
    "MARUTI.NS": "Auto", "TATAMOTORS.NS": "Auto", "M&M.NS": "Auto",
    "RELIANCE.NS": "Energy", "ONGC.NS": "Energy", "POWERGRID.NS": "Energy"
}
panel_df["Sector"] = panel_df["Firm"].map(sector_map).fillna("Other")

# Returns by Sector
plt.figure(figsize=(10, 4))
panel_df.groupby("Sector")["Log_Return"].mean().sort_values().plot(kind="bar", color='teal')
plt.title("Average Returns by Sector")
plt.ylabel("Daily Mean Return")
plt.show()

## 6. Panel Data Regressions
We transition the dataset into a standard multi-index panel format (`Firm`, `Date`) and estimate regressions. Fixed Effects controls for time-invariant individual unobserved characteristics.

In [ ]:
panel_df["Date"] = pd.to_datetime(panel_df["Date"])
panel_data = panel_df.set_index(["Firm", "Date"])

exog_vars = ["Volatility", "Momentum", "MA_Ratio", "RSI", "Market_Return", "VIX"]
X = sm.add_constant(panel_data[exog_vars])
y = panel_data["Log_Return"]

print("--- 1. POOLED OLS ESTIMATESATION ---")
pooled_results = PanelOLS(y, X, entity_effects=False, time_effects=False).fit()
print(pooled_results.summary.tables[1])

print("--- 2. FIXED EFFECTS (ENTITY EFFECTS) ESTIMATESATION ---")
fe_results = PanelOLS(y, X, entity_effects=True, time_effects=False).fit()
print(fe_results.summary.tables[1])

print("--- 3. RANDOM EFFECTS ESTIMATESATION ---")
re_results = RandomEffects(y, X).fit()
print(re_results.summary.tables[1])

print("--- 4. FIRST DIFFERENCE ESTIMATESATION ---")
fd_df = panel_data[["Log_Return"] + exog_vars].groupby(level=0).diff().dropna()
y_fd = fd_df["Log_Return"]
X_fd = sm.add_constant(fd_df[exog_vars])
fd_results = PanelOLS(y_fd, X_fd, entity_effects=False, time_effects=False).fit()
print(fd_results.summary.tables[1])